# Guardrails AI — NimbusPay Customer-Support Bot
**Five-part video series.** Each Part maps to one video.

*Scenario:* NimbusPay, a fictional fintech. A support bot that handles refunds, account questions, and complaints. Fintech is deliberate — PII, competitors, topic-restriction, and a compliance disclaimer all fall out of the domain naturally.

| Part | Video | Content |
|------|-------|---------|
| 0 | pre-video | Setup: installs, tokens, model config |
| 1 | V1 | Why & What + Input / Output Guards |
| 2 | V2 | Risk Mitigation: Hub validators + Custom validators |
| 3 | V3 | Enforcing Structure: RAIL → Pydantic + under-the-hood |
| 4 | V4 | The Guard Object: call vs parse, reask, history |
| — | — | Capstone + "what it costs" |

**Models (Groq via LiteLLM):**
- `groq/llama-3.3-70b-versatile` — primary. Stable production workhorse, strong instruction-following.
- `groq/llama-3.1-8b-instant` — streaming stage only; fast enough that token-by-token validation is visibly snappy on screen.

> Pin to **stable** model IDs. Groq rotates its lineup — preview-tier models (e.g. `kimi-k2`, deprecated March 2026) can vanish mid-course and break the notebook on first run.


---
## Part 0 · Setup

Shared pre-video cells. Run once; re-running is a no-op.

> ⚠ **Gotchas for the narration:**
> - Guardrails talks to Groq **through LiteLLM** (`model="groq/..."`).
> - There is **no `jsonformer`** on a remote provider — structuring is prompt-based + reask, not constrained decoding.
> - `ToxicLanguage` and `RestrictToTopic` **download ML models on install**. The first run is slow — warn before hitting this cell on camera.


In [ ]:
# Pinned for reproducibility. Re-run is a no-op once installed.
%pip install -q "guardrails-ai==0.6.6" "litellm>=1.55.0" "groq>=0.13.0" "pydantic>=2.7"


### Hub token & API key

Two pieces of friction, both called out on camera:
1. **`guardrails configure`** needs a **Hub token** (free, from hub.guardrailsai.com).  This is the *token-as-friction* moment — you can't `guardrails hub install` anything  without it.
2. **`GROQ_API_KEY`** comes from the environment, never hardcoded — a small  security-hygiene nod that fits the course.


In [ ]:
import os, getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")

if not os.environ.get("GUARDRAILS_TOKEN"):
    os.environ["GUARDRAILS_TOKEN"] = getpass.getpass(
        "GUARDRAILS_TOKEN (free at hub.guardrailsai.com): "
    )

print("Keys loaded:",
      bool(os.environ.get("GROQ_API_KEY")),
      bool(os.environ.get("GUARDRAILS_TOKEN")))


In [ ]:
!guardrails configure --token $GUARDRAILS_TOKEN --disable-metrics --disable-remote-inferencing


### Hub validator installs

> ⚠ **Slow cell on first run.** `toxic_language` and `restrict_to_topic` fetch ML  models (~200–400 MB). Run it once and grab a coffee. Subsequent runs are instant.


In [ ]:
!guardrails hub install hub://guardrails/detect_pii --quiet
!guardrails hub install hub://guardrails/competitor_check --quiet
!guardrails hub install hub://guardrails/restrict_to_topic --quiet
!guardrails hub install hub://guardrails/toxic_language --quiet
!guardrails hub install hub://guardrails/gibberish_text --quiet
print("Hub validators ready.")


In [ ]:
# Shared config — referenced by every part below.
BOT_MODEL    = "groq/llama-3.3-70b-versatile"   # primary
STREAM_MODEL = "groq/llama-3.1-8b-instant"       # streaming stage only

SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)
print("Models:", BOT_MODEL, "|", STREAM_MODEL)


---
## Part 1 · Why & What + Input / Output Guards  *(Video 1)*

### Two functions Guardrails performs at runtime

1. **Input guard** — validates the *user's prompt* before it reaches the model.  Catches PII the user accidentally pasted, injection attempts, off-topic requests.
2. **Output guard** — validates the *model's reply* before it reaches the user.  Catches PII leaks, competitor slips, toxic language, malformed structure.

> *Runtime vs pre-deployment:* Guardrails AI is a **runtime** layer — it wraps live  calls. Pre-deployment scanning (static analysis of prompts/configs) is a different  category of tool.

We'll start with the naive bot and then wrap it on **both sides**.


In [ ]:
import litellm

def naive_bot(user_msg: str) -> str:
    """Plain Groq call — no guards, no structure."""
    resp = litellm.completion(
        model=BOT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content

# Happy path works — this is the baseline we harden from here.
print(naive_bot("Hi, how long do refunds usually take?"))


### Wrapping the naive bot — input guard

The input guard runs on the **user's message** *before* it hits the model. If it fires, we never spend tokens on a bad prompt.

Here we use `DetectPII` with `on_fail=EXCEPTION` — if the user accidentally pastes a card number in their question, block the call entirely.


In [ ]:
from guardrails import Guard, OnFailAction
from guardrails.hub import DetectPII
from guardrails.errors import ValidationError

# Input guard — validate the user's prompt
input_guard = Guard().use(
    DetectPII,
    pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
    on_fail=OnFailAction.EXCEPTION,
)

def bot_with_input_guard(user_msg: str) -> str:
    try:
        input_guard.parse(user_msg)          # raises if PII found
    except ValidationError as e:
        return f"[INPUT BLOCKED — PII detected before model call]"
    return naive_bot(user_msg)

# Safe prompt — passes through
print("SAFE:", bot_with_input_guard("How long do refunds take?"))
# Blocked prompt — card number in the user's message
print("RISKY:", bot_with_input_guard("My card 4111 1111 1111 1111 was charged twice."))


### Wrapping the naive bot — output guard

The output guard runs on the **model's reply** *before* the user sees it. With `on_fail=FIX`, PII is anonymised rather than blocked — the conversation continues, the leak doesn't.


In [ ]:
# Output guard — scrub PII from the model's reply
output_guard = Guard().use(
    DetectPII,
    pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
    on_fail=OnFailAction.FIX,
)

def bot_with_output_guard(user_msg: str) -> str:
    reply = naive_bot(user_msg)
    return output_guard.parse(reply).validated_output

# A message that tempts the bot to mirror back PII
test_msg = (
    "My email is confirm@nimbuspay.io and my card ending 1234 "
    "was double-charged. What's my refund status?"
)
raw   = naive_bot(test_msg)
clean = bot_with_output_guard(test_msg)

print("RAW  :", raw[:220])
print()
print("CLEAN:", clean[:220])


---
## Part 2 · Risk Mitigation: Validators + Hub + Custom  *(Video 2)*

**Validators are pre-built risk measures.** The Hub is the library. Each validator focuses on one failure mode; you compose them on a Guard.

### `on_fail` actions — match the action to the cost of the failure

| Action | Effect | Use when |
|--------|--------|----------|
| `FIX` | Anonymise / patch the output | Low-harm; keep conversation flowing |
| `FILTER` | Remove the offending sentence(s) | Medium-harm; rest of reply is fine |
| `REFRAIN` | Return empty output | High-harm; don't surface anything |
| `EXCEPTION` | Raise `ValidationError` | Hard stop; handle in caller |
| `NOOP` | Record but don't change | Logging / observability only |

> *Pedagogical note:* We're using `Guard` minimally here — just as the wrapper.  We'll go deep on the Guard object itself in Part 4.


#### Hub validator 1 · `DetectPII` — `on_fail=FIX`

In [ ]:
from guardrails.hub import DetectPII
from guardrails import Guard, OnFailAction

pii_guard = Guard().use(
    DetectPII,
    pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
    on_fail=OnFailAction.FIX,
)

leaked = (
    "Thanks! I can see the charge on card 4111 1111 1111 1111 "
    "tied to john.doe@example.com — we'll refund it within 3 days."
)
res = pii_guard.parse(leaked)
print("BEFORE:", leaked)
print("AFTER :", res.validated_output)


#### Hub validator 2 · `CompetitorCheck` — `on_fail=FILTER`

In [ ]:
from guardrails.hub import CompetitorCheck

comp_guard = Guard().use(
    CompetitorCheck,
    competitors=["Razorpay", "PayU", "Stripe", "PayPal"],
    on_fail=OnFailAction.FILTER,
)

slip = (
    "You could try NimbusPay's instant payout. Honestly Razorpay is also popular, "
    "and Stripe has great docs. NimbusPay support is 24/7."
)
res = comp_guard.parse(slip)
print("BEFORE:", slip)
print("AFTER :", res.validated_output)


#### Hub validator 3 · `ToxicLanguage` — `on_fail=REFRAIN`

In [ ]:
from guardrails.hub import ToxicLanguage

tox_guard = Guard().use(
    ToxicLanguage,
    threshold=0.5,
    validation_method="sentence",
    on_fail=OnFailAction.REFRAIN,
)

clean_reply = "I understand the frustration — let me pull up your account and sort this out."
toxic_reply = "Honestly you're an idiot and this is entirely your own stupid fault."

for label, txt in [("clean", clean_reply), ("toxic", toxic_reply)]:
    out = tox_guard.parse(txt).validated_output
    print(f"{label:5} -> {out!r}")


#### Hub validator 4 · `RestrictToTopic` — `on_fail=EXCEPTION`

In [ ]:
from guardrails.hub import RestrictToTopic
from guardrails.errors import ValidationError

topic_guard = Guard().use(
    RestrictToTopic,
    valid_topics=["payments", "refunds", "account support", "billing"],
    invalid_topics=["poetry", "politics", "medical advice"],
    disable_classifier=True,   # use the LLM zero-shot path (no local model needed)
    disable_llm=False,
    llm_callable=BOT_MODEL,
    on_fail=OnFailAction.EXCEPTION,
)

poem = (
    "Sure! Here's a poem about the ocean: "
    "The waves roll in, the tide pulls out, the gulls cry overhead..."
)
try:
    topic_guard.parse(poem)
    print("PASSED (unexpected)")
except ValidationError as e:
    print("BLOCKED off-topic ->", str(e)[:200])


#### Hub validator 5 · `GibberishText` — `on_fail=EXCEPTION`

Catches nonsense output — important if you're piping replies into a downstream system that expects coherent text.


In [ ]:
from guardrails.hub import GibberishText

gibberish_guard = Guard().use(
    GibberishText,
    threshold=0.5,
    validation_method="sentence",
    on_fail=OnFailAction.EXCEPTION,
)

legit = "Your refund will be processed within 3–5 business days."
junk  = "Zyxwvutsrqponmlkjihgfedcba qwertyuiop asdfghjkl."

for label, txt in [("legit", legit), ("junk", junk)]:
    try:
        gibberish_guard.parse(txt)
        print(f"{label} -> PASSED")
    except ValidationError as e:
        print(f"{label} -> BLOCKED: {str(e)[:120]}")


### Custom validators — when the Hub doesn't cover your rule

Two forms: **function form** (stateless, `@register_validator`) and **class form** (parameterised, subclass of `Validator`). Use the class form when your validator needs arguments or mutable state.

**NimbusPay rules:**
- (a) *Disclaimer:* any refund answer must carry "subject to verification".
- (b) *Cap:* refund amounts above `threshold` need human sign-off.


In [ ]:
import re
from guardrails.validators import (
    Validator, register_validator, PassResult, FailResult, ValidationResult
)

# ── (a) Function form: stateless disclaimer check ──────────────────────────
@register_validator(name="nimbus/refund-disclaimer", data_type="string")
def refund_disclaimer(value: str, metadata: dict) -> ValidationResult:
    mentions_refund = "refund" in value.lower()
    has_disclaimer  = "subject to verification" in value.lower()
    if mentions_refund and not has_disclaimer:
        return FailResult(
            error_message="Refund answer is missing the 'subject to verification' disclaimer.",
            fix_value=value.rstrip(".") + ". All refunds are subject to verification.",
        )
    return PassResult()


# ── (b) Class form: parameterised refund cap ───────────────────────────────
@register_validator(name="nimbus/max-refund-claim", data_type="string")
class MaxRefundClaim(Validator):
    """Flags any dollar amount above `threshold` — those need human sign-off."""

    def __init__(self, threshold: float = 500.0, on_fail=None):
        super().__init__(on_fail=on_fail, threshold=threshold)
        self.threshold = threshold

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        amounts = [
            float(a.replace(",", ""))
            for a in re.findall(r"\$\s?([\d,]+(?:\.\d+)?)", value)
        ]
        over = [a for a in amounts if a > self.threshold]
        if over:
            return FailResult(
                error_message=(
                    f"Promised refund {over} exceeds ${self.threshold:.0f} "
                    "auto-approve cap — needs human sign-off."
                ),
            )
        return PassResult()

print("Custom validators registered.")


In [ ]:
# ── Demonstrate both custom validators ─────────────────────────────────────

# (a) Disclaimer — auto-fixed on fail
disc_guard = Guard().use(refund_disclaimer, on_fail=OnFailAction.FIX)
out = disc_guard.parse("Yes, we will process your refund within 3 business days.")
print("disclaimer fix ->", out.validated_output)

print()

# (b) Refund cap — refrained when over threshold
cap_guard = Guard().use(MaxRefundClaim, threshold=500.0, on_fail=OnFailAction.REFRAIN)
under = cap_guard.parse("We'll refund you $120 today.").validated_output
over  = cap_guard.parse("We'll refund the full $4,300 immediately.").validated_output
print("under cap ($120) ->", repr(under))
print("over cap  ($4300)->", repr(over))   # None — refrained


---
## Part 3 · Enforcing Structure: RAIL → Pydantic + Under the Hood  *(Video 3)*

### RAIL — the original blueprint

Guardrails AI's original approach used `.rail` XML files to define the expected output schema, validators, and prompts in one document. **RAIL XML is legacy** — you'll still see it in older examples and docs, but the modern path is Pydantic.

> ⚠ **Cautionary aside for the recording:** the old RAIL evaluator used Python's  `eval()` to coerce types. That CVE is a good example of why "prompt injection can  become code execution" is a real threat in AI frameworks — not just theory.

### Pydantic as the modern expression

`Guard.for_pydantic(Model)` takes a Pydantic model and does two things:
1. Injects the schema as instructions into the system prompt.
2. Parses and validates the model's reply back into the Pydantic type.

For models that support **function/tool calling** (Groq's llama-3.3 does), Guardrails can use the native structured-output path instead of prompt injection — same Guard call, different wire format. We'll see both below.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from guardrails import Guard

class SupportResponse(BaseModel):
    answer: str = Field(description="The reply shown to the customer.")
    category: Literal["refund", "account", "complaint", "other"] = Field(
        description="Ticket routing category."
    )
    needs_human: bool = Field(description="True if a human agent must follow up.")
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="Customer's apparent sentiment."
    )

struct_guard = Guard.for_pydantic(SupportResponse)

res = struct_guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "I was double-charged on my coffee this morning!"},
    ],
    temperature=0.2,
)

print(res.validated_output)
print("\ntype:", type(res.validated_output))


### Under the hood — what Guardrails actually sent to Groq

`guard.history` records every call. Inspecting it reveals the **schema injected into the system prompt** — you can see exactly what Groq received, including the JSON format instructions Guardrails added. This is the payoff of this video.


In [ ]:
# Inspect the prompt that was sent to Groq
history = struct_guard.history

if history:
    last_call = history[-1]
    iteration = last_call.iterations[0]
    msgs = iteration.inputs.messages

    print(f"Messages sent to Groq ({len(msgs)} total):")
    for msg in msgs:
        role    = msg.get("role", "?").upper()
        content = msg.get("content", "")
        print(f"\n[{role}]")
        print(content[:800])
        print("---")
else:
    print("No history yet — run the cell above first.")


### Function-calling path

For models that expose a tool/function API, Guardrails can route the schema through native structured outputs instead of injecting format instructions into the prompt. The Guard call is **identical** — the difference is entirely under the hood. Inspect `guard.history` after this call to compare the wire format.


In [ ]:
# Same guard, same call signature — Guardrails picks the path based on the model's
# advertised capabilities (reported via LiteLLM's model_info).
res_fc = struct_guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "How do I change my NimbusPay PIN?"},
    ],
    temperature=0.2,
)
print("Structured output:", res_fc.validated_output)

# Compare history: did Guardrails inject schema into the prompt, or use tool calling?
last = struct_guard.history[-1]
iteration = last.iterations[0]
used_tool_calling = any(
    "tool" in str(msg).lower() or "function" in str(msg).lower()
    for msg in iteration.inputs.messages
)
print("\nTool-calling path used:", used_tool_calling)
print("Prompt injection used  :", not used_tool_calling)


---
## Part 4 · The Guard Object: Call vs Parse + Reask  *(Video 4)*

The Guard is the **orchestrator**: it wraps model calls, runs validation pipelines, tracks history, and handles failure recovery. Three flows to know:

| Flow | When to use |
|------|-------------|
| `guard(model=…, messages=[…])` | **Call flow** — Guard calls the model *and* validates |
| `guard.parse(text)` | **Parse flow** — validate text you already have |
| `num_reasks=N` on the call | **Reask** — let the model self-correct on validation failure |


### Call flow

In [ ]:
from guardrails import Guard, OnFailAction
from guardrails.hub import DetectPII, CompetitorCheck
from pydantic import BaseModel, Field
from typing import Literal

# Build a guard that structures *and* validates in one call
call_guard = (
    Guard.for_pydantic(SupportResponse)
    .use(DetectPII, pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS"],
         on_fail=OnFailAction.FIX, on="answer")
    .use(CompetitorCheck, competitors=["Razorpay", "PayU", "Stripe", "PayPal"],
         on_fail=OnFailAction.FILTER, on="answer")
)

# Single .call() → model invoked, reply validated, Pydantic object returned
res = call_guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "My card 4111 1111 1111 1111 was charged incorrectly."},
    ],
    temperature=0.2,
)
print("Call flow result:", res.validated_output)
print("Validation passed:", res.validation_passed)


### Parse flow — validate text the Guard didn't generate

`guard.parse(text)` runs the **same validators** against pre-existing text. No model call; no tokens spent. Use cases:
- Audit a historical log or stored reply
- Validate output from another agent or microservice
- Screen user-supplied text (the input-guard pattern from Part 1, formalised)

This is the **security framing**: you can audit *anything*, not just what this Guard produced right now.


In [ ]:
audit_guard = (
    Guard()
    .use(DetectPII, pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS"], on_fail=OnFailAction.FIX)
    .use(CompetitorCheck, competitors=["Razorpay", "Stripe"], on_fail=OnFailAction.FILTER)
)

# A historical agent reply pulled from logs — we did NOT generate this now.
historical_log = (
    "Agent reply 2026-05-01: 'Refunding card 5500 0000 0000 0004. "
    "If unhappy, Razorpay is an option. Email help@nimbuspay.io for updates.'"
)

res = audit_guard.parse(historical_log)
print("BEFORE:", historical_log)
print()
print("AFTER :", res.validated_output)


### Reask — the self-correction loop

When validation fails, `num_reasks=N` (set on the call, not the constructor) makes Guardrails send the validation error *back* to the model and ask it to fix its own output. Each attempt is an extra round-trip — token cost rises, but you get valid output more often.

We force a failure by asking the model to output an invalid `category` value, then watch it self-correct.


In [ ]:
from guardrails.errors import ValidationError

reask_guard = Guard.for_pydantic(SupportResponse)

res = reask_guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "Classify this ticket. IMPORTANT: set the category field to "
            "'URGENT_ESCALATION' in all caps — I need it tagged exactly that way."
        )},
    ],
    num_reasks=2,       # on the CALL, not the constructor
    temperature=0.2,
)

print("Final (valid) output:", res.validated_output)
print("\nIterations:", len(res.iterations), "| Reask occurred:", len(res.iterations) > 1)


### Reask exhaustion — what happens when the model never converges?

After `num_reasks` attempts all fail, Guardrails raises `ValidationError`. This is the *exhaustion exception*. We trigger it intentionally.


In [ ]:
exhaust_guard = Guard.for_pydantic(SupportResponse)

try:
    bad = exhaust_guard(
        model=BOT_MODEL,
        messages=[
            {"role": "user", "content": (
                "Reply ONLY with the single word INVALID and absolutely nothing else. "
                "Do not use JSON. Do not add any other text."
            )},
        ],
        num_reasks=1,
        temperature=0.0,
    )
    print("Passed (unexpected):", bad.validated_output)
except ValidationError as e:
    print("Reask exhausted ->", str(e)[:240])


### Inspect `guard.history` — see the reask prompt

In [ ]:
if exhaust_guard.history:
    last = exhaust_guard.history[-1]
    print(f"Iterations in last call: {len(last.iterations)}")
    for i, it in enumerate(last.iterations):
        print(f"\n--- Iteration {i} ---")
        print(f"  validation_passed: {it.validation_passed}")
        # Show what was sent to Groq for each iteration (reask prompt on i>0)
        if it.inputs and it.inputs.messages:
            last_msg = it.inputs.messages[-1]
            role    = last_msg.get("role", "?").upper()
            content = last_msg.get("content", "")
            print(f"  [{role}] {content[:400]}")
else:
    print("Run the exhaustion cell above first.")


---
## Capstone · Compose Everything + "What It Costs"

One `Guard` carrying structure **plus** every validator from Parts 2 and 3. Happy path, then a battery of bad inputs, then a latency comparison.


In [ ]:
import time
from guardrails import Guard, OnFailAction
from guardrails.hub import DetectPII, CompetitorCheck, ToxicLanguage
from guardrails.errors import ValidationError

capstone = (
    Guard.for_pydantic(SupportResponse)
    .use(DetectPII,
         pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
         on_fail=OnFailAction.FIX, on="answer")
    .use(CompetitorCheck,
         competitors=["Razorpay", "PayU", "Stripe", "PayPal"],
         on_fail=OnFailAction.FILTER, on="answer")
    .use(ToxicLanguage,
         threshold=0.5, validation_method="sentence",
         on_fail=OnFailAction.FIX, on="answer")
    .use(refund_disclaimer,
         on_fail=OnFailAction.FIX, on="answer")
    .use(MaxRefundClaim,
         threshold=500.0, on_fail=OnFailAction.REFRAIN, on="answer")
)

def run_capstone(user_msg: str):
    return capstone(
        model=BOT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        num_reasks=2,
        temperature=0.2,
    )

# Happy path
happy = run_capstone("How long do refunds take?")
print("Happy path:", happy.validated_output)


In [ ]:
# Battery of challenging inputs
tests = [
    ("normal",       "How do I reset my NimbusPay PIN?"),
    ("refund ask",   "Please refund my $80 grocery charge."),
    ("competitor",   "Should I just switch to Razorpay instead?"),
    ("toxic+cap",    "You useless idiots! Refund my $9,000 RIGHT NOW."),
    ("pii in reply", "My card 4111 1111 1111 1111 was charged — refund it."),
]

print(f"{'SCENARIO':16} | {'ANSWER (truncated)':52} | CATEGORY  | HUMAN")
print("-" * 105)
for scenario, msg in tests:
    try:
        out = run_capstone(msg).validated_output
        if out and out.answer:
            snippet = repr(out.answer[:46])
            print(f"{scenario:16} | {snippet:52} | {out.category:9} | {out.needs_human}")
        else:
            print(f"{scenario:16} | {'<refrained — over cap or toxic>':52} |           |")
    except ValidationError as e:
        print(f"{scenario:16} | {'<blocked: ' + str(e)[:40] + '>':52} |           |")
    except Exception as e:
        print(f"{scenario:16} | {'<error: ' + type(e).__name__ + '>':52} |           |")


In [ ]:
# ── Latency & reask overhead ──────────────────────────────────────────────

def timed(fn, *args):
    t0 = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - t0

q = "How long do refunds take?"
_, t_naive    = timed(naive_bot, q)
_, t_capstone = timed(lambda x: run_capstone(x).validated_output, q)

print(f"naive call   : {t_naive:6.2f}s")
print(f"full guard   : {t_capstone:6.2f}s")
print(f"overhead     : {t_capstone - t_naive:+6.2f}s  ({(t_capstone / t_naive - 1) * 100:+.0f}%)")
print()
print("Reask cost: each reask = 1 extra round-trip + ~(input + output) tokens.")
print("Budget accordingly: 2 reasks on a 500-token call can triple token spend.")


### When to *skip* Guardrails AI

The decision-framework takeaway — reach for it when the cost is justified:

**Use it when** bad output has real downside:
- PII / compliance exposure on a fintech or healthcare bot
- Structured output that *must* parse (downstream system depends on it)
- Brand-safety on user-facing text (competitor slips, toxic replies)
- Auditing external text (parse flow — no extra model calls)

**Skip it / go lighter when:**
- Latency is critical and output is low-risk (internal tooling, dev assistants)
- You control structure another way — constrained decoding on a local model with  `jsonformer` / `outlines` doesn't need reask at all
- A single regex or a cheap `if` covers the one rule you actually have

**Right-size `on_fail`:**
- `fix`/`filter` keep the conversation flowing; use them for recoverable failures
- `refrain`/`exception` are for hard stops; use them when surfacing the bad output  is genuinely worse than surfacing nothing

**The loop back:** every cell above maps to one module node. The cost table is the framework node that tells you *which* of them a given feature actually needs.
